# Code for the four figures
Each section is independent. Run the cell for the figure you want.
These reproduce the original Lin-versus-five-term-model comparisons.

## Baseline Jacobian eigenvalues
figure_1_eigenvalues.py

In [ ]:
"""Figure 1: Baseline Jacobian eigenvalues.

Each marker is a Jacobian eigenvalue at the stated model baseline.
Negative real parts imply local exponential stability; positive parts imply instability.

Requires numpy, scipy, matplotlib. Run this file by itself.
"""
from pathlib import Path
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Save images without needing a desktop plot window.
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from scipy.integrate import solve_ivp

# Lin parameter order:      (a, b, c, d, g, h).
# Modified parameter order: (a, k, c, g, h).
# Received restraint is d in Lin and h in the modified model.
# Lin is the original six-term comparator, not the newer matched comparator.
PL = (1, 1, 1, 1, 1, 1)
PS = (2, 0.5, 1, 0.5, 0.3)
NAMES = ["Wood", "Fire", "Earth", "Metal", "Water"]
COL = ["#26845b", "#c74641", "#a05aa1", "#454b57", "#347db6"]
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = BASE_DIR / "generated_figures"
OUT.mkdir(parents=True, exist_ok=True)
R = {}
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 200})


def save(name, fig):
    """Write both a PNG image and an editable vector SVG."""
    fig.savefig(OUT / (name + ".png"), bbox_inches="tight")
    fig.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.close(fig)

def lin(t, x, a, b, c, d, g, h):
    """Lin six-term equation. Node order: Wood, Fire, Earth, Metal, Water."""
    return x * (a - b * x + c * np.roll(x, 1) - d * np.roll(x, 2) - g * np.roll(x, -1) - h * np.roll(x, -2))

def sat(t, x, a, k, c, g, h):
    """Canonical five-term DM_(5,2). No additional loss terms are included."""
    return a * x - k * x * x + c * np.roll(x, 1) / (1 + x) - g * x / (1 + np.roll(x, -2)) - h * x * np.roll(x, 2)

def eq_l(p):
    """Positive synchronous level of Lin; NaN means no such equilibrium."""
    (a, b, c, d, g, h) = p
    M = b + d + g + h - c
    return a / M if M > 0 else np.nan

def roots_s(p):
    """All positive synchronous levels of the five-term model, sorted."""
    (a, k, c, g, h) = p
    A = k + h
    D = (A - a) ** 2 + 4 * A * (a + c - g)
    if D < 0:
        return []
    return sorted(set([v for v in ((a - A - np.sqrt(D)) / (2 * A), (a - A + np.sqrt(D)) / (2 * A)) if v > 0]))

def eq_s(p):
    """Select the upper positive synchronous branch used in the figures."""
    r = roots_s(p)
    return max(r) if r else np.nan

def jl(x, p):
    """Lin Jacobian at a five-component state; p=(a,b,c,d,g,h)."""
    (a, b, c, d, g, h) = p
    J = np.zeros((5, 5))
    for i in range(5):
        J[i, i] = a - 2 * b * x[i] + c * x[(i - 1) % 5] - d * x[(i - 2) % 5] - g * x[(i + 1) % 5] - h * x[(i + 2) % 5]
        for (off, v) in [(-1, c), (-2, -d), (1, -g), (2, -h)]:
            J[i, (i + off) % 5] = v * x[i]
    return J

def js(x, p):
    """Five-term modified Jacobian; p=(a,k,c,g,h)."""
    (a, k, c, g, h) = p
    J = np.zeros((5, 5))
    for i in range(5):
        s = 1 + x[i]
        ta = x[(i + 2) % 5]
        J[i, i] = a - 2 * k * x[i] - c * x[(i - 1) % 5] / s ** 2 - g / (1 + ta) - h * x[(i - 2) % 5]
        J[i, (i - 1) % 5] = c / s
        J[i, (i + 2) % 5] = g * x[i] / (1 + ta) ** 2
        J[i, (i - 2) % 5] = -h * x[i]
    return J

def eigs(m, p, x=None):
    """Compute all five Jacobian eigenvalues at the selected balanced level."""
    if x is None:
        x = eq_l(p) if m == 'lin' else eq_s(p)
    if not np.isfinite(x):
        return np.full(5, np.nan)
    return np.linalg.eigvals((jl if m == 'lin' else js)(np.full(5, x), p))

def alpha(m, p):
    """Largest eigenvalue real part; NaN if positive balance is absent."""
    return np.max(eigs(m, p).real)

# Calculate the data and draw this figure.
R['baseline'] = {}
(fig, ax) = plt.subplots(figsize=(7.5, 4.5), layout='constrained')
ax.axvspan(0, 1, color='#f9e8e6')
ax.axvline(0, c='0.3', lw=1)
ax.axhline(0, c='0.8', lw=0.7)
for (m, p, col, mk, lab) in [('lin', PL, '#bb453d', 'o', 'Lin: all coefficients = 1'), ('sat', PS, '#24628d', 'x', 'Modified: (2, 0.5, 1, 0.5, 0.3)')]:
    es = eigs(m, p)
    xe = eq_l(p) if m == 'lin' else eq_s(p)
    R['baseline'][m] = {'xstar': xe, 'alpha': float(max(es.real)), 'eigenvalues': [[float(z.real), float(z.imag)] for z in sorted(es, key=lambda z: -z.real)]}
    ax.scatter(es.real, es.imag, c=col, marker=mk, s=80, label=lab)
ax.set(xlabel='Re $\\lambda$', ylabel='Im $\\lambda$', xlim=(-2.6, 0.8), ylim=(-1.3, 1.3), title='Local stability at the stated baseline parameters')
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.18)
save('01_baseline_eigenvalues', fig)

# Save the underlying counts/numerical checks beside the graph.
(OUT / "figure_1_results.json").write_text(json.dumps(R, indent=2))
print(json.dumps(R, indent=2))
print(f"Saved figure files to {OUT}")


## Responses to the two Wood disturbances
figure_2_disturbances.py

In [ ]:
"""Figure 2: Responses to the two Wood disturbances.

Start at each model's equilibrium; multiply Wood by 1.05 or 0.30.
All runs reach t=120; modified-model panels display only t<=12.
The RK45 rerun checks the plotted DOP853 trajectories.

Requires numpy, scipy, matplotlib. Run this file by itself.
"""
from pathlib import Path
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Save images without needing a desktop plot window.
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from scipy.integrate import solve_ivp

# Lin parameter order:      (a, b, c, d, g, h).
# Modified parameter order: (a, k, c, g, h).
# Received restraint is d in Lin and h in the modified model.
# Lin is the original six-term comparator, not the newer matched comparator.
PL = (1, 1, 1, 1, 1, 1)
PS = (2, 0.5, 1, 0.5, 0.3)
NAMES = ["Wood", "Fire", "Earth", "Metal", "Water"]
COL = ["#26845b", "#c74641", "#a05aa1", "#454b57", "#347db6"]
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = BASE_DIR / "generated_figures"
OUT.mkdir(parents=True, exist_ok=True)
R = {}
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 200})


def save(name, fig):
    """Write both a PNG image and an editable vector SVG."""
    fig.savefig(OUT / (name + ".png"), bbox_inches="tight")
    fig.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.close(fig)

def lin(t, x, a, b, c, d, g, h):
    """Lin six-term equation. Node order: Wood, Fire, Earth, Metal, Water."""
    return x * (a - b * x + c * np.roll(x, 1) - d * np.roll(x, 2) - g * np.roll(x, -1) - h * np.roll(x, -2))

def sat(t, x, a, k, c, g, h):
    """Canonical five-term DM_(5,2). No additional loss terms are included."""
    return a * x - k * x * x + c * np.roll(x, 1) / (1 + x) - g * x / (1 + np.roll(x, -2)) - h * x * np.roll(x, 2)

def eq_l(p):
    """Positive synchronous level of Lin; NaN means no such equilibrium."""
    (a, b, c, d, g, h) = p
    M = b + d + g + h - c
    return a / M if M > 0 else np.nan

def roots_s(p):
    """All positive synchronous levels of the five-term model, sorted."""
    (a, k, c, g, h) = p
    A = k + h
    D = (A - a) ** 2 + 4 * A * (a + c - g)
    if D < 0:
        return []
    return sorted(set([v for v in ((a - A - np.sqrt(D)) / (2 * A), (a - A + np.sqrt(D)) / (2 * A)) if v > 0]))

def eq_s(p):
    """Select the upper positive synchronous branch used in the figures."""
    r = roots_s(p)
    return max(r) if r else np.nan

def integrate(m, p, x0, T=120, method='DOP853', n=3001):
    """Full five-node integration on 3,001 sampled times; check solver success."""
    r = solve_ivp(lin if m == 'lin' else sat, (0, T), x0, args=p, method=method, rtol=1e-10, atol=1e-12, t_eval=np.linspace(0, T, n))
    if not r.success:
        raise RuntimeError(r.message)
    return r

# Calculate the data and draw this figure.
(fig, axes) = plt.subplots(2, 2, figsize=(10.4, 6.3), layout='constrained')
R['perturbations'] = []
for (col, (m, p)) in enumerate([('lin', PL), ('sat', PS)]):
    xe = eq_l(p) if m == 'lin' else eq_s(p)
    for (row, factor) in enumerate([1.05, 0.3]):
        x = np.full(5, xe)
        x[0] *= factor
        s = integrate(m, p, x)
        check = integrate(m, p, x, method='RK45')
        R['perturbations'].append({'model': m, 'wood_factor': factor, 'initial_norm': float(np.linalg.norm(x - xe)), 'final_norm': float(np.linalg.norm(s.y[:, -1] - xe)), 'minimum_over_run': float(s.y.min()), 'RK45_DOP853_max_abs_difference': float(np.max(np.abs(s.y - check.y)))})
        ax = axes[row, col]
        for i in range(5):
            ax.plot(s.t, s.y[i] / xe, c=COL[i], lw=1.05, label=NAMES[i])
        ax.axhline(1, c='0.6', ls='--', lw=0.8)
        ax.set(xlabel='Model time', ylabel='$x_i / x^*$', title=f"{('Lin' if m == 'lin' else 'Modified')}: Wood {('+5%' if row == 0 else 'reduced to 30%')}")
        ax.set_xlim(0, 120 if m == 'lin' else 12)
fig.legend(NAMES, loc='outside lower center', ncol=5, frameon=False)
fig.suptitle('Response to a one-component initial disturbance', fontsize=13)
save('02_perturbation_recovery', fig)

# Save the underlying counts/numerical checks beside the graph.
(OUT / "figure_2_results.json").write_text(json.dumps(R, indent=2))
print(json.dumps(R, indent=2))
print(f"Saved figure files to {OUT}")


## Two-parameter stability maps
figure_3_stability_maps.py

In [ ]:
"""Figure 3: Two-parameter stability maps.

For the displayed master grid: c in [0.3,1.6], restraint in [0.1,1.0].
Lin tuple = (1.5,0.65,c,restraint,0.3,0.3).
Modified tuple = (1.5,0.65,c,0.3,restraint).
At each grid point, select positive balance, compute eigenvalues, then color
by the largest real part. Count stability using alpha < -1e-9.
The earlier grid is also computed for the supplementary numerical comparison;
only the master grid is plotted. Missing equilibria are gray, not unstable.
Color magnitudes are clipped at a shared percentile, not the data itself.

Requires numpy, scipy, matplotlib. Run this file by itself.
"""
from pathlib import Path
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Save images without needing a desktop plot window.
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from scipy.integrate import solve_ivp

# Lin parameter order:      (a, b, c, d, g, h).
# Modified parameter order: (a, k, c, g, h).
# Received restraint is d in Lin and h in the modified model.
# Lin is the original six-term comparator, not the newer matched comparator.
PL = (1, 1, 1, 1, 1, 1)
PS = (2, 0.5, 1, 0.5, 0.3)
NAMES = ["Wood", "Fire", "Earth", "Metal", "Water"]
COL = ["#26845b", "#c74641", "#a05aa1", "#454b57", "#347db6"]
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = BASE_DIR / "generated_figures"
OUT.mkdir(parents=True, exist_ok=True)
R = {}
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 200})


def save(name, fig):
    """Write both a PNG image and an editable vector SVG."""
    fig.savefig(OUT / (name + ".png"), bbox_inches="tight")
    fig.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.close(fig)

def lin(t, x, a, b, c, d, g, h):
    """Lin six-term equation. Node order: Wood, Fire, Earth, Metal, Water."""
    return x * (a - b * x + c * np.roll(x, 1) - d * np.roll(x, 2) - g * np.roll(x, -1) - h * np.roll(x, -2))

def sat(t, x, a, k, c, g, h):
    """Canonical five-term DM_(5,2). No additional loss terms are included."""
    return a * x - k * x * x + c * np.roll(x, 1) / (1 + x) - g * x / (1 + np.roll(x, -2)) - h * x * np.roll(x, 2)

def eq_l(p):
    """Positive synchronous level of Lin; NaN means no such equilibrium."""
    (a, b, c, d, g, h) = p
    M = b + d + g + h - c
    return a / M if M > 0 else np.nan

def roots_s(p):
    """All positive synchronous levels of the five-term model, sorted."""
    (a, k, c, g, h) = p
    A = k + h
    D = (A - a) ** 2 + 4 * A * (a + c - g)
    if D < 0:
        return []
    return sorted(set([v for v in ((a - A - np.sqrt(D)) / (2 * A), (a - A + np.sqrt(D)) / (2 * A)) if v > 0]))

def eq_s(p):
    """Select the upper positive synchronous branch used in the figures."""
    r = roots_s(p)
    return max(r) if r else np.nan

def jl(x, p):
    """Lin Jacobian at a five-component state; p=(a,b,c,d,g,h)."""
    (a, b, c, d, g, h) = p
    J = np.zeros((5, 5))
    for i in range(5):
        J[i, i] = a - 2 * b * x[i] + c * x[(i - 1) % 5] - d * x[(i - 2) % 5] - g * x[(i + 1) % 5] - h * x[(i + 2) % 5]
        for (off, v) in [(-1, c), (-2, -d), (1, -g), (2, -h)]:
            J[i, (i + off) % 5] = v * x[i]
    return J

def js(x, p):
    """Five-term modified Jacobian; p=(a,k,c,g,h)."""
    (a, k, c, g, h) = p
    J = np.zeros((5, 5))
    for i in range(5):
        s = 1 + x[i]
        ta = x[(i + 2) % 5]
        J[i, i] = a - 2 * k * x[i] - c * x[(i - 1) % 5] / s ** 2 - g / (1 + ta) - h * x[(i - 2) % 5]
        J[i, (i - 1) % 5] = c / s
        J[i, (i + 2) % 5] = g * x[i] / (1 + ta) ** 2
        J[i, (i - 2) % 5] = -h * x[i]
    return J

def eigs(m, p, x=None):
    """Compute all five Jacobian eigenvalues at the selected balanced level."""
    if x is None:
        x = eq_l(p) if m == 'lin' else eq_s(p)
    if not np.isfinite(x):
        return np.full(5, np.nan)
    return np.linalg.eigvals((jl if m == 'lin' else js)(np.full(5, x), p))

def alpha(m, p):
    """Largest eigenvalue real part; NaN if positive balance is absent."""
    return np.max(eigs(m, p).real)

# Calculate the data and draw this figure.
R['maps'] = {}
maps = []
for (name, N, cs, rs) in [('early', 90, np.linspace(0.05, 2.5, 90), np.linspace(0.05, 2, 90)), ('master', 90, np.linspace(0.3, 1.6, 90), np.linspace(0.1, 1, 90))]:
    arr = []
    stats = {}
    for m in ['lin', 'sat']:
        z = np.full((N, N), np.nan)
        for (iy, r) in enumerate(rs):
            for (ix, c) in enumerate(cs):
                p = ((1, 1, c, 1, 1, r) if m == 'lin' else (2, 0.5, c, 0.5, r)) if name == 'early' else (1.5, 0.65, c, r, 0.3, 0.3) if m == 'lin' else (1.5, 0.65, c, 0.3, r)
                z[iy, ix] = alpha(m, p)
        valid = np.isfinite(z)
        stable = valid & (z < -1e-09)
        stats[m] = {'stable': int(stable.sum()), 'valid': int(valid.sum()), 'total': N * N, 'fraction_all': float(stable.mean()), 'fraction_valid': float(stable.sum() / valid.sum()), 'min': float(np.nanmin(z)), 'max': float(np.nanmax(z))}
        arr.append(z)
    R['maps'][name] = stats
    if name == 'master':
        maps = arr
        mapcs = cs
        maprs = rs
(fig, axes) = plt.subplots(1, 2, figsize=(11, 4.7), layout='constrained')
vm = max(0.5, np.nanpercentile(np.abs(np.concatenate([z.ravel() for z in maps])), 90))
cmap = plt.get_cmap('RdBu_r').copy()
cmap.set_bad('#b9bec6')
for (ax, m, z) in zip(axes, ['lin', 'sat'], maps):
    im = ax.pcolormesh(mapcs, maprs, z, cmap=cmap, norm=TwoSlopeNorm(0, -vm, vm), shading='nearest')
    if np.nanmin(z) < 0 < np.nanmax(z):
        ax.contour(mapcs, maprs, z, levels=[0], colors='k', linewidths=1.4)
    st = R['maps']['master'][m]
    ax.set(xlabel='Nourishment coefficient c', ylabel='Received restraint d (Lin)' if m == 'lin' else 'Received restraint h (modified)', title=f"{('Lin' if m == 'lin' else 'Modified')}: {100 * st['fraction_all']:.1f}% of all grid points stable")
    ax.text(0.02, 0.025, f"Among existing positive equilibria: {100 * st['fraction_valid']:.1f}%", transform=ax.transAxes, fontsize=8, bbox={'fc': 'white', 'ec': 'none', 'alpha': 0.9})
    fig.colorbar(im, ax=ax, extend='both', label='$\\max\\mathrm{Re}\\,\\lambda$ (color limits clipped)')
fig.suptitle('Balanced-equilibrium stability on the master notebook grid', fontsize=13)
fig.legend(handles=[Patch(color='#b9bec6', label='No positive balanced equilibrium'), plt.Line2D([], [], c='k', label='Interpolated zero contour')], loc='outside lower center', ncol=2, frameon=False, fontsize=8)
save('03_corrected_stability_maps', fig)

# Save the underlying counts/numerical checks beside the graph.
(OUT / "figure_3_results.json").write_text(json.dumps(R, indent=2))
print(json.dumps(R, indent=2))
print(f"Saved figure files to {OUT}")


## Stability frequencies under three sampling schemes
figure_4_sampling.py

In [ ]:
"""Figure 4: Stability frequencies under three sampling schemes.

Draw 1,200 accepted parameter sets per scenario with seed 3.
Reset the generator for each scenario, preserving the original draw order.
Uniform: all six conceptual coefficients uniform on [0.1,2].
Ordered: same distribution conditional on SELF > SHENG > RESTR > TGT.
Tiered: independent draws in the explicit TIERS intervals below.
Each draw is paired across models; CHILD has no counterpart in the modified model.
Preserve TIERS insertion order to reproduce the same seeded results.
Left bars condition on positive-balance existence; right bars use all draws.
The original sampling calculation uses alpha < 0; near-zero cases should be
examined separately if you change these parameters or sampling ranges.

Requires numpy, scipy, matplotlib. Run this file by itself.
"""
from pathlib import Path
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Save images without needing a desktop plot window.
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from scipy.integrate import solve_ivp

# Lin parameter order:      (a, b, c, d, g, h).
# Modified parameter order: (a, k, c, g, h).
# Received restraint is d in Lin and h in the modified model.
# Lin is the original six-term comparator, not the newer matched comparator.
PL = (1, 1, 1, 1, 1, 1)
PS = (2, 0.5, 1, 0.5, 0.3)
NAMES = ["Wood", "Fire", "Earth", "Metal", "Water"]
COL = ["#26845b", "#c74641", "#a05aa1", "#454b57", "#347db6"]
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = BASE_DIR / "generated_figures"
OUT.mkdir(parents=True, exist_ok=True)
R = {}
plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 200})


def save(name, fig):
    """Write both a PNG image and an editable vector SVG."""
    fig.savefig(OUT / (name + ".png"), bbox_inches="tight")
    fig.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.close(fig)

def lin(t, x, a, b, c, d, g, h):
    """Lin six-term equation. Node order: Wood, Fire, Earth, Metal, Water."""
    return x * (a - b * x + c * np.roll(x, 1) - d * np.roll(x, 2) - g * np.roll(x, -1) - h * np.roll(x, -2))

def sat(t, x, a, k, c, g, h):
    """Canonical five-term DM_(5,2). No additional loss terms are included."""
    return a * x - k * x * x + c * np.roll(x, 1) / (1 + x) - g * x / (1 + np.roll(x, -2)) - h * x * np.roll(x, 2)

def eq_l(p):
    """Positive synchronous level of Lin; NaN means no such equilibrium."""
    (a, b, c, d, g, h) = p
    M = b + d + g + h - c
    return a / M if M > 0 else np.nan

def roots_s(p):
    """All positive synchronous levels of the five-term model, sorted."""
    (a, k, c, g, h) = p
    A = k + h
    D = (A - a) ** 2 + 4 * A * (a + c - g)
    if D < 0:
        return []
    return sorted(set([v for v in ((a - A - np.sqrt(D)) / (2 * A), (a - A + np.sqrt(D)) / (2 * A)) if v > 0]))

def eq_s(p):
    """Select the upper positive synchronous branch used in the figures."""
    r = roots_s(p)
    return max(r) if r else np.nan

def jl(x, p):
    """Lin Jacobian at a five-component state; p=(a,b,c,d,g,h)."""
    (a, b, c, d, g, h) = p
    J = np.zeros((5, 5))
    for i in range(5):
        J[i, i] = a - 2 * b * x[i] + c * x[(i - 1) % 5] - d * x[(i - 2) % 5] - g * x[(i + 1) % 5] - h * x[(i + 2) % 5]
        for (off, v) in [(-1, c), (-2, -d), (1, -g), (2, -h)]:
            J[i, (i + off) % 5] = v * x[i]
    return J

def js(x, p):
    """Five-term modified Jacobian; p=(a,k,c,g,h)."""
    (a, k, c, g, h) = p
    J = np.zeros((5, 5))
    for i in range(5):
        s = 1 + x[i]
        ta = x[(i + 2) % 5]
        J[i, i] = a - 2 * k * x[i] - c * x[(i - 1) % 5] / s ** 2 - g / (1 + ta) - h * x[(i - 2) % 5]
        J[i, (i - 1) % 5] = c / s
        J[i, (i + 2) % 5] = g * x[i] / (1 + ta) ** 2
        J[i, (i - 2) % 5] = -h * x[i]
    return J

def eigs(m, p, x=None):
    """Compute all five Jacobian eigenvalues at the selected balanced level."""
    if x is None:
        x = eq_l(p) if m == 'lin' else eq_s(p)
    if not np.isfinite(x):
        return np.full(5, np.nan)
    return np.linalg.eigvals((jl if m == 'lin' else js)(np.full(5, x), p))

def alpha(m, p):
    """Largest eigenvalue real part; NaN if positive balance is absent."""
    return np.max(eigs(m, p).real)

# Calculate the data and draw this figure.
TIERS = {'SELF': (1, 2), 'SHENG': (0.5, 1.5), 'SAT': (0.3, 1), 'RESTR': (0.2, 0.8), 'TGT': (0.1, 0.5), 'CHILD': (0.1, 0.5)}

def draw(r, scheme):
    if scheme == 'tiered':
        return {k: r.uniform(*v) for (k, v) in TIERS.items()}
    while True:
        (S, A, SH, RE, TG, CH) = r.uniform(0.1, 2, 6)
        if scheme == 'full_order' and (not S > SH > RE > TG):
            continue
        return dict(SELF=S, SAT=A, SHENG=SH, RESTR=RE, TGT=TG, CHILD=CH)

def to_p(m, d):
    return (d['SELF'], d['SAT'], d['SHENG'], d['RESTR'], d['CHILD'], d['TGT']) if m == 'lin' else (d['SELF'], d['SAT'], d['SHENG'], d['TGT'], d['RESTR'])

def wilson(k, n):
    z = 1.96
    p = k / n
    den = 1 + z * z / n
    ctr = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return [ctr - h, ctr + h]
R['monte_carlo'] = {}
for scheme in ['uniform', 'full_order', 'tiered']:
    r = np.random.default_rng(3)
    v = {m: [] for m in ['lin', 'sat']}
    for _ in range(1200):
        d = draw(r, scheme)
        for m in v:
            v[m].append(alpha(m, to_p(m, d)))
    ss = {}
    common = np.isfinite(v['lin']) & np.isfinite(v['sat'])
    for (m, z) in v.items():
        z = np.array(z)
        va = np.isfinite(z)
        st = va & (z < 0)
        (ns, nv) = (int(st.sum()), int(va.sum()))
        ss[m] = {'stable': ns, 'valid': nv, 'total': 1200, 'conditional': ns / nv, 'unconditional': ns / 1200, 'wilson_conditional': wilson(ns, nv), 'both_exist_n': int(common.sum()), 'both_exist_stable_fraction': float(np.mean(z[common] < 0))}
    R['monte_carlo'][scheme] = ss
(fig, axes) = plt.subplots(1, 2, figsize=(10.7, 4.6), layout='constrained')
for (ax, key, title) in zip(axes, ['conditional', 'unconditional'], ['Among draws with positive balance', 'Among all 1,200 draws']):
    x = np.arange(3)
    for (m, offset, col, lab) in [('lin', -0.18, '#c05b53', 'Lin'), ('sat', 0.18, '#347aa2', 'Modified')]:
        vals = [100 * R['monte_carlo'][s][m][key] for s in R['monte_carlo']]
        bars = ax.bar(x + offset, vals, 0.36, color=col, label=lab)
        for (b, val) in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, val + 1.5, f'{val:.1f}%', ha='center', fontsize=8)
    ax.set_xticks(x, ['Uniform', 'Ordered', 'Tiered'])
    ax.set(ylim=(0, 111), ylabel='Draws with locally stable positive balance (%)', title=title)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle('Stability frequencies depend on the specified sampling scheme', fontsize=13)
save('04_sampling_denominators', fig)

# Save the underlying counts/numerical checks beside the graph.
(OUT / "figure_4_results.json").write_text(json.dumps(R, indent=2))
print(json.dumps(R, indent=2))
print(f"Saved figure files to {OUT}")
